# Task 6 (Part 1): Generation for SFT, DPO-HH, and Persona Follow-ups (`align-generate`)

In [ ]:
!nvidia-smi
import os, shutil, pathlib

input_root = pathlib.Path('/kaggle/input')
src_dir = next(input_root.rglob('submission_adapter.py')).parent
work_repo = pathlib.Path('/tmp/repo')
if work_repo.exists():
    shutil.rmtree(work_repo)
shutil.copytree(src_dir, work_repo)
os.chdir(work_repo)
print('Found code dataset at:', src_dir)
print('COMMIT:', (work_repo / 'COMMIT.txt').read_text().strip())


In [ ]:
!python -m pip uninstall -y -q torchao || true
!python -m pip install -q "transformers==4.57.6" "datasets>=2.18,<5" "trl==1.16.0" "peft>=0.11.0" huggingface_hub accelerate tqdm pyyaml sentencepiece safetensors
!python -m pip install -q -e . --no-deps
import torch, transformers, trl, peft
print('PyTorch:', torch.__version__, 'CUDA:', torch.cuda.is_available(), 'GPUs:', torch.cuda.device_count())
print('Transformers:', transformers.__version__, 'TRL:', trl.__version__, 'PEFT:', peft.__version__)


In [ ]:
# 1. 3-prompt sanity checks FIRST (locked greedy settings: max_new_tokens=512, repetition_penalty=1.0)
!CUDA_VISIBLE_DEVICES=0 PYTHONPATH=. python -m src.align.generate \
  --model_name_or_path outputs/sft/best \
  --eval_set_path eval_sets/alpaca_eval_300_seed42.json \
  --output_path /kaggle/working/outputs/generations/sanity_3_alpaca_sft.json \
  --generator_name qwen2.5-0.5b-sft \
  --dataset_name alpaca_eval \
  --max_new_tokens 512 --repetition_penalty 1.0 --no_repeat_ngram_size 0 --limit 3 --batch_size 3

!CUDA_VISIBLE_DEVICES=0 PYTHONPATH=. python -m src.align.generate \
  --model_name_or_path outputs/sft/best \
  --eval_set_path eval_sets/hh_rlhf_eval_300_seed42.json \
  --output_path /kaggle/working/outputs/generations/sanity_3_hh_sft.json \
  --generator_name qwen2.5-0.5b-sft \
  --dataset_name hh_rlhf \
  --max_new_tokens 512 --repetition_penalty 1.0 --no_repeat_ngram_size 0 --limit 3 --batch_size 3

!CUDA_VISIBLE_DEVICES=0 PYTHONPATH=. python -m src.align.generate \
  --model_name_or_path outputs/dpo_hh/best \
  --eval_set_path eval_sets/hh_rlhf_eval_300_seed42.json \
  --output_path /kaggle/working/outputs/generations/sanity_3_hh_dpo_hh.json \
  --generator_name qwen2.5-0.5b-dpo-hh \
  --dataset_name hh_rlhf \
  --max_new_tokens 512 --repetition_penalty 1.0 --no_repeat_ngram_size 0 --limit 3 --batch_size 3


In [ ]:
# 2. SFT vs DPO-HH on HH-RLHF 300 + SFT on AlpacaEval 300 (greedy, repetition_penalty=1.0)
!CUDA_VISIBLE_DEVICES=0 PYTHONPATH=. python -m src.align.generate \
  --model_name_or_path outputs/sft/best \
  --eval_set_path eval_sets/hh_rlhf_eval_300_seed42.json \
  --output_path /kaggle/working/outputs/generations/hh_sft.json \
  --generator_name qwen2.5-0.5b-sft \
  --dataset_name hh_rlhf \
  --max_new_tokens 512 --repetition_penalty 1.0 --no_repeat_ngram_size 0 --batch_size 16

!CUDA_VISIBLE_DEVICES=0 PYTHONPATH=. python -m src.align.generate \
  --model_name_or_path outputs/dpo_hh/best \
  --eval_set_path eval_sets/hh_rlhf_eval_300_seed42.json \
  --output_path /kaggle/working/outputs/generations/hh_dpo_hh.json \
  --generator_name qwen2.5-0.5b-dpo-hh \
  --dataset_name hh_rlhf \
  --max_new_tokens 512 --repetition_penalty 1.0 --no_repeat_ngram_size 0 --batch_size 16

!CUDA_VISIBLE_DEVICES=0 PYTHONPATH=. python -m src.align.generate \
  --model_name_or_path outputs/sft/best \
  --eval_set_path eval_sets/alpaca_eval_300_seed42.json \
  --output_path /kaggle/working/outputs/generations/alpaca_sft.json \
  --generator_name qwen2.5-0.5b-sft \
  --dataset_name alpaca_eval \
  --max_new_tokens 512 --repetition_penalty 1.0 --no_repeat_ngram_size 0 --batch_size 16


In [ ]:
# 3. Apply >10% max_new_tokens rule:
#    - Base Qwen2.5-0.5B hit 30.33% (>10%) on AlpacaEval 300 and 25.33% (>10%) on HH-RLHF 300,
#      so generate Base and SFT with repetition_penalty=1.1 on AlpacaEval 300 (and HH-RLHF 300).
#    - If either hh_sft or hh_dpo_hh > 10% (or unconditionally so all rp=1.1 files exist),
#      generate hh_sft_rp11.json and hh_dpo_hh_rp11.json with repetition_penalty=1.1.
import json, subprocess

with open('/kaggle/working/outputs/generations/hh_sft_summary.json') as f:
    hh_sft_sum = json.load(f)
with open('/kaggle/working/outputs/generations/hh_dpo_hh_summary.json') as f:
    hh_dpo_sum = json.load(f)
with open('/kaggle/working/outputs/generations/alpaca_sft_summary.json') as f:
    alp_sft_sum = json.load(f)

print(f"Greedy (rp=1.0) hit_max_new_tokens -> hh_sft={hh_sft_sum['pct_hit_max_new_tokens']}%, hh_dpo_hh={hh_dpo_sum['pct_hit_max_new_tokens']}%, alpaca_sft={alp_sft_sum['pct_hit_max_new_tokens']}%")

# Base vs SFT on AlpacaEval 300 already triggered by Base (30.33% > 10%); also needed if SFT vs DPO-UF triggers
!CUDA_VISIBLE_DEVICES=0 PYTHONPATH=. python -m src.align.generate \
  --model_name_or_path Qwen/Qwen2.5-0.5B \
  --eval_set_path eval_sets/alpaca_eval_300_seed42.json \
  --output_path /kaggle/working/outputs/generations/alpaca_base_rp11.json \
  --generator_name qwen2.5-0.5b-base-rp11 \
  --dataset_name alpaca_eval \
  --max_new_tokens 512 --repetition_penalty 1.1 --no_repeat_ngram_size 0 --batch_size 16

!CUDA_VISIBLE_DEVICES=0 PYTHONPATH=. python -m src.align.generate \
  --model_name_or_path outputs/sft/best \
  --eval_set_path eval_sets/alpaca_eval_300_seed42.json \
  --output_path /kaggle/working/outputs/generations/alpaca_sft_rp11.json \
  --generator_name qwen2.5-0.5b-sft-rp11 \
  --dataset_name alpaca_eval \
  --max_new_tokens 512 --repetition_penalty 1.1 --no_repeat_ngram_size 0 --batch_size 16

# Check SFT vs DPO-HH on HH-RLHF 300 (plus Base vs SFT on HH-RLHF 300 where Base=25.33% > 10%)
!CUDA_VISIBLE_DEVICES=0 PYTHONPATH=. python -m src.align.generate \
  --model_name_or_path Qwen/Qwen2.5-0.5B \
  --eval_set_path eval_sets/hh_rlhf_eval_300_seed42.json \
  --output_path /kaggle/working/outputs/generations/hh_base_rp11.json \
  --generator_name qwen2.5-0.5b-base-rp11 \
  --dataset_name hh_rlhf \
  --max_new_tokens 512 --repetition_penalty 1.1 --no_repeat_ngram_size 0 --batch_size 16

!CUDA_VISIBLE_DEVICES=0 PYTHONPATH=. python -m src.align.generate \
  --model_name_or_path outputs/sft/best \
  --eval_set_path eval_sets/hh_rlhf_eval_300_seed42.json \
  --output_path /kaggle/working/outputs/generations/hh_sft_rp11.json \
  --generator_name qwen2.5-0.5b-sft-rp11 \
  --dataset_name hh_rlhf \
  --max_new_tokens 512 --repetition_penalty 1.1 --no_repeat_ngram_size 0 --batch_size 16

if hh_sft_sum['pct_hit_max_new_tokens'] > 10.0 or hh_dpo_sum['pct_hit_max_new_tokens'] > 10.0:
    print('SFT vs DPO-HH triggered >10% rule; generating hh_dpo_hh_rp11.json with repetition_penalty=1.1')
    subprocess.run([
        'python', '-m', 'src.align.generate',
        '--model_name_or_path', 'outputs/dpo_hh/best',
        '--eval_set_path', 'eval_sets/hh_rlhf_eval_300_seed42.json',
        '--output_path', '/kaggle/working/outputs/generations/hh_dpo_hh_rp11.json',
        '--generator_name', 'qwen2.5-0.5b-dpo-hh-rp11',
        '--dataset_name', 'hh_rlhf',
        '--max_new_tokens', '512', '--repetition_penalty', '1.1', '--no_repeat_ngram_size', '0', '--batch_size', '16'
    ], check=True)


In [ ]:
# 4. Persona Follow-ups:
#    (a) Regenerate base SmolLM2 and 3-epoch LoRA adapter with repetition_penalty=1.1
!CUDA_VISIBLE_DEVICES=0 PYTHONPATH=. python -m src.align.generate \
  --model_name_or_path HuggingFaceTB/SmolLM2-360M-Instruct \
  --eval_set_path eval_sets/persona_style_eval.json \
  --output_path /kaggle/working/outputs/lora_persona_followup/persona_base_eval_rp11.json \
  --generator_name smollm2-360m-instruct-base-rp11 \
  --dataset_name persona_style_eval \
  --use_chat_template \
  --max_new_tokens 512 --repetition_penalty 1.1 --no_repeat_ngram_size 0 --batch_size 8

!CUDA_VISIBLE_DEVICES=0 PYTHONPATH=. python -m src.align.generate \
  --model_name_or_path HuggingFaceTB/SmolLM2-360M-Instruct \
  --lora_adapter_path outputs/lora_persona/best_adapter \
  --eval_set_path eval_sets/persona_style_eval.json \
  --output_path /kaggle/working/outputs/lora_persona_followup/persona_lora_3ep_eval_rp11.json \
  --generator_name smollm2-360m-instruct-ramsay-lora-3ep-rp11 \
  --dataset_name persona_style_eval \
  --use_chat_template \
  --max_new_tokens 512 --repetition_penalty 1.1 --no_repeat_ngram_size 0 --batch_size 8

#    (b) Retrain LoRA for 1 epoch (configs/lora_persona_1ep.yaml) and generate with both greedy (rp=1.0) and penalty (rp=1.1)
!CUDA_VISIBLE_DEVICES=0 PYTHONPATH=. python -m src.align.train_lora_dpo --config configs/lora_persona_1ep.yaml

!CUDA_VISIBLE_DEVICES=0 PYTHONPATH=. python -m src.align.generate \
  --model_name_or_path /kaggle/working/outputs/lora_persona_1ep/best_merged \
  --eval_set_path eval_sets/persona_style_eval.json \
  --output_path /kaggle/working/outputs/lora_persona_1ep/persona_lora_1ep_eval.json \
  --generator_name smollm2-360m-instruct-ramsay-lora-1ep \
  --dataset_name persona_style_eval \
  --use_chat_template \
  --max_new_tokens 512 --repetition_penalty 1.0 --no_repeat_ngram_size 0 --batch_size 8

!CUDA_VISIBLE_DEVICES=0 PYTHONPATH=. python -m src.align.generate \
  --model_name_or_path /kaggle/working/outputs/lora_persona_1ep/best_merged \
  --eval_set_path eval_sets/persona_style_eval.json \
  --output_path /kaggle/working/outputs/lora_persona_1ep/persona_lora_1ep_eval_rp11.json \
  --generator_name smollm2-360m-instruct-ramsay-lora-1ep-rp11 \
  --dataset_name persona_style_eval \
  --use_chat_template \
  --max_new_tokens 512 --repetition_penalty 1.1 --no_repeat_ngram_size 0 --batch_size 8

import shutil
shutil.rmtree('/kaggle/working/outputs/lora_persona_1ep/best_merged', ignore_errors=True)
shutil.rmtree('/kaggle/working/outputs/lora_persona_1ep/trl_checkpoints', ignore_errors=True)
